# Preprocessing of the RTS data to prepare for the Issue Index creation

In [2]:
import os
import json
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import csv

### Directory Structure: `/mnt/project_impresso/original/RTS`

The RTS directory contains radio content organized by program/show names. The filesystem structure is organized as follows:

```
/mnt/project_impresso/original/RTS/
│
├── News Programs (Journaux)
│   ├── j_mat/                    (morning news)  
│   │   ├── audio                       (all PM3 audio files)
│   │   ├── stt                         (all stt and xml files containing the text related to each audio file)
│   │   ├── ExpXml-20251106-122951.xml  (xml files with the metadata relating the MP3 audio and XML files for each listing)
│   │   ...  
│   │   └── ExpXml-20251113-171933.xml                   
│   ├── j_midi/                   (midday news)
│   ├── j13h/                     (1 PM news)
│   ├── j13h2/                    (1 PM news variant)
│   ├── j_soir/                   (evening news)
│   └── j_nuit/                   (night news)
│
...
│
└── Other Programs
    ├── petitdej/                 (breakfast)
    ├── ana_media/                (media analysis)
    ├── geneve_info/              (Geneva information)
    └── enquest/                  (enquête/investigation)
```

Each directory contains media files (audio recordings and associated metadata) for that specific radio program.

In [3]:
base_dir = "/mnt/project_impresso/original/RTS"

audios_subdir = 'audio'
asr_subdir = 'stt'
metadata_file_start = "ExpXml"

## Process an example of metadata xml file to extract the contents

In [4]:
example_program = "causerie_uni"

example_program_dir = os.path.join(base_dir, example_program)

ex_meta_files = [os.path.join(example_program_dir,f) for f in os.listdir(example_program_dir) if f.startswith(metadata_file_start)]
ex_meta_files

['/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-131700.xml',
 '/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-132007.xml']

In [25]:
with open(ex_meta_files[0], "r", encoding="utf-8") as f:
    raw_xml = f.read()

xml_doc = BeautifulSoup(raw_xml, "xml")
xml_doc

<?xml version="1.0" encoding="utf-8"?>
<DOCUMENTS><DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><RO

In [6]:
docs = xml_doc.find_all("DOCUMENT")
docs 

[<DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><ROLE>privat-docent ? l'Universit? de Fribourg</ROLE

In [7]:
for child in docs[0].find_all(recursive=False):
    print(f"name: {child.name}")
    print(f"attrs: {child.attrs}")
    print(f"text: {child.get_text(strip=True)}")

name: CLSID
attrs: {}
text: {D2593F4E-C887-4E48-8982-5BD08BA4DAE0}
name: OID
attrs: {}
text: {3267F04D-657F-4DCB-BCB0-44B2B6C2682D}
name: LOGIN
attrs: {}
text: 
name: TITLE
attrs: {}
text: Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg
name: HIERARCHY
attrs: {'title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg", 'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}', 'depth': '---', 'current': '*'}
text: 
name: SEQUENCE
attrs: {}
text: 1
name: BROADCAST
attrs: {}
text: Causerie universitaire
name: DOCUMENTTYPE
attrs: {}
text: Parl?
name: GEOGRAPHICALDESCRIPTORS
attrs: {}
text: Pologne
name: HIERARCHYLEVEL
attrs: {}
text: Sujet
name: MODIFIEDBY
attrs: {}
text: albrecjo
name: MODIFIEDON
attrs: {}
text: 25.05.2022 03:56:53
name: HISTORY
attrs: {}
text: Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B
name: PARTICIPANTS
attrs: {}
text: Cros, 

In [12]:
# define more intuitive names
simple_fields_renaming = {
    "CLSID": "cls_ID",
    "OID": "OID",
    "LOGIN": "login",
    "TITLE": "broadcast_episode_title",
    "SEQUENCE": "sequence", 
    "BROADCAST": "broadcast_program_name",
    "DOCUMENTTYPE": "document_type",
    "HIERARCHYLEVEL": "hierarchy_level",
    "MODIFIEDBY": "modified_by",
    "MODIFIEDON": "modified_on",
    "HISTORY": "physical_support_history",
    "PRODUCTIONTYPE": "production_type",
    "RECORDINGPLACE": "recording_place",
    "RIGHTSNOTES": "rights_notes",
    "RIGHTSSTATUS": "rights_status",
    "SERIESTITLE": "series_title",
    "SUMMARY": "content_summary",
    "WORKFLOWSTATUS": "workflow_status",
    "ASSEMBLYSTATUS": "assembly_status",
    "LIVE": "live",
    "MODULATIONTYPE": "modulation_type",
    "WORKDURATION": "work_duration",
    "WORKDURATIONCOMPL": "work_duration_compl",
}

list_fields_renaming = {
    'GEOGRAPHICALDESCRIPTORS': 'geographical_descriptors',
    'PERSONDESCRIPTORS': 'person_descriptors',
    'THEMATICALDESCRIPTORS': 'thematical_descriptors',
    'RIGHTSUSAGEPOSSIBILITIES': 'rights_usage_possibilities',
    'PROGRAMMES': 'radio_channels',
    'SUBDOMAINS': 'subdomains',
    'RECORDINGDATES': 'recording_dates',
    'FIRSTBROADCASTDATES': 'first_broadcast_dates'
}

support_keys = ['spt_clsid', 'spt_oid',
                'spt_title',
                'spt_isdigital',
                'spt_filename',
                'spt_cataloguing_status',
                'spt_source',
                'spt_unit_duration']

doc_keys = ["alias", "date_str", "stt_filename", "mp3_filenames", "stripped_OID", "exact_date", "broadcast_date"] + list(simple_fields_renaming.values()) + list(list_fields_renaming.values()) + ["supports", "spt_filenames", "participants"]

In [13]:
# Parse DOCUMENT elements into structured dictionaries
def parse_docs_in_xml(xml_doc, program, all_alias_stt, all_alias_audios, doc_keys=doc_keys, simple_fields_map=simple_fields_renaming, list_fields_map=list_fields_renaming):
    """
    Extract all DOCUMENT elements from XML into a list of dictionaries.
    Handles nested structures: PARTICIPANTS, GEOGRAPHICALDESCRIPTORS, 
    THEMATICALDESCRIPTORS, SUPPORTS, RECORDINGDATES, etc.
    """
    documents = []
    
    # Find all DOCUMENT elements
    doc_elements = xml_doc.find_all('DOCUMENT')
    skipped = 0
    
    print(f"\nStarting extracting {len(doc_elements)} documents for program {program}")
    for doc_idx, doc_elem in enumerate(doc_elements):
        doc_dict = {
            "alias": program,
            "date_str": None,
            "stt_filename": None,
            "mp3_filenames": None
        }
        
        # Extract simple text fields
        
        
        for og_field, renamed_field in simple_fields_map.items():
            elem = doc_elem.find(og_field)
            if elem:
                text = elem.get_text(strip=True)
                doc_dict[renamed_field] = text
                if og_field=='OID':
                    # the text is actually "{oid}", remove start and end brackets
                    doc_dict['stripped_OID'] = text[1:-1] if text else None
            else:
                doc_dict[renamed_field] = None
        
        if "stripped_OID" not in doc_dict or not doc_dict['stripped_OID']:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} because it's missing an OID!!")
            skipped +=1
            continue


        stt_file = f"{doc_dict['stripped_OID']}_STT.xml"
        if stt_file not in all_alias_stt:
            stt_text_file = f"{doc_dict['stripped_OID']}_STT.txt"
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its XML file!! (stt exists: {stt_text_file in all_alias_stt})")
            skipped +=1
            continue
        else:
            doc_dict['stt_filename'] = stt_file

        # Extract PARTICIPANTS (list of dicts)
        participants = []
        for participant in doc_elem.find_all('PARTICIPANT'):
            name_elem = participant.find('NAME')
            function_elem = participant.find('FUNCTION')
            role_elem = participant.find('ROLE')
            participants.append({
                'name': name_elem.get_text(strip=True) if name_elem else None,
                'function': function_elem.get_text(strip=True) if function_elem else None,
                'role': role_elem.get_text(strip=True) if role_elem else None
            })
        if participants:
            doc_dict['participants'] = participants
        
        # Extract list fields (DESCRIPTORS, PROGRAMMES, SUBDOMAINS, etc.)
        list_fields = {
            'GEOGRAPHICALDESCRIPTORS': 'GEOGRAPHICALDESCRIPTOR',
            'PERSONDESCRIPTORS': 'PERSONDESCRIPTOR',
            'THEMATICALDESCRIPTORS': 'THEMATICALDESCRIPTOR',
            'RIGHTSUSAGEPOSSIBILITIES': 'RIGHTSUSAGEPOSSIBILITY',
            'PROGRAMMES': 'PROGRAMME',
            'SUBDOMAINS': 'SUBDOMAIN',
            'RECORDINGDATES': 'RECORDINGDATE',
            'FIRSTBROADCASTDATES': 'FIRSTBROADCASTDATE'
        }
        
        for container_name, renamed_field in list_fields_map.items():
            container = doc_elem.find(container_name)
            if container:
                doc_dict[renamed_field] = [elem.get_text(strip=True) for elem in container.find_all(list_fields[container_name])]
            else:
                # always define fields, set them to None if not defined
                doc_dict[renamed_field] = None

        date_strings = None
        broadcast_date = None
        # extract the date from first_braodcast_dates
        if doc_dict["first_broadcast_dates"]:
            date_strings = [d for rd in doc_dict["first_broadcast_dates"] for d in rd.split(" - ") if d != "__/__/____"]
            broadcast_date = True 
            
        if not date_strings and doc_dict["recording_dates"]:
            print(f"Did not find any date for doc with OID {doc_dict['OID']}. Trying to use the recording date. doc_dict: {doc_dict}")
            date_strings = [d for rd in doc_dict["recording_dates"] for d in rd.split(" - ") if d != "__/__/____" and "Avant" not in d and "Apr?s" not in d]
            broadcast_date = False

        # process the dates extracted
        if not date_strings:
            print(f"WARNING! MISSING DATE FOR DOC WITH OID {doc_dict['OID']}!! \nDocument: {doc_dict}, \noriginal: {doc_elem}")
            if doc_dict['stripped_OID'] == "5D740FA1-1A90-4878-A7E2-40C43C0F6430":
                print(f"Found and setting manually the correct date for {doc_dict['OID']}: 10/04/1989")
                doc_dict['date_str'] = "10/04/1989"
                doc_dict['exact_date'] = True
                doc_dict['broadcast_date'] = True
            #doc_dict['day'] = None
        else:
            # reformat each date and keep the earliest
            dates = []
            exact_dates = []
            for s in date_strings:
                exact = True
                if "__" in s:
                    old_s = s
                    s = s.replace("__", "01")
                    exact = False
                    print(f"The date for document with OID {doc_dict['OID']} was invalid ({old_s}) - changed it to {s}")
                if s.startswith('~'):
                    final_d = datetime.strptime(s[1:],  "%d/%m/%Y")
                else:   
                    final_d = datetime.strptime(s,  "%d/%m/%Y")
                dates.append(final_d)
                if exact:
                    exact_dates.append(final_d)
                

                    
            #dates = [datetime.strptime(s[1:] if s.startswith('~') else s, "%d/%m/%Y") for s in date_strings]
            #doc_dict['year'] = min(dates).year
            #doc_dict['month'] = min(dates).month
            #doc_dict['day'] = min(dates).day
            chosen_date = min(dates)
            doc_dict['date_str'] = chosen_date.strftime('%d/%m/%Y')
            doc_dict['exact_date'] = chosen_date in exact_dates
            doc_dict['broadcast_date'] = broadcast_date
            
        
        # Extract SUPPORTS (audio files and metadata)
        supports = []
        mp3_filenames = []

        for support in doc_elem.find_all('SUPPORT'):
            support_dict = {
                'spt_clsid': support.find('CLSID').get_text(strip=True) if support.find('CLSID') else None,
                'spt_oid': support.find('OID').get_text(strip=True) if support.find('OID') else None,
                'spt_title': support.find('TITLE').get_text(strip=True) if support.find('TITLE') else None,
                'spt_isdigital': support.find('ISDIGITAL').get_text(strip=True) if support.find('ISDIGITAL') else None,
                'spt_filename': support.find('FILENAME').get_text(strip=True) if support.find('FILENAME') else None,
                'spt_cataloguing_status': support.find('CATALOGUINGSTATUS').get_text(strip=True) if support.find('CATALOGUINGSTATUS') else None,
                'spt_source': support.find('SOURCE').get_text(strip=True) if support.find('SOURCE') else None,
                'spt_unit_duration': support.find('UNITDURATION').get_text(strip=True) if support.find('UNITDURATION') else None
            }

            if support_dict['spt_filename']:
                # remove '.wav' if it's in the filename
                spt_filename = support_dict['spt_filename'].replace('.wav', "")
                # populate the list of mp3 filenames with any mp3 file which has a matching filename
                mp3_filenames.extend([audio for audio in all_alias_audios if spt_filename in audio])

            supports.append(support_dict)

        if not supports or not mp3_filenames:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its audio MP3 file!!")
            skipped +=1
            continue

        doc_dict['supports'] = supports
        doc_dict['spt_filenames'] = [s['spt_filename'] for s in supports]
        doc_dict['mp3_filenames'] = mp3_filenames

        # before adding to the list of docs, check it has all keys, and setting any missing one to None
        for k in doc_keys:
            if k not in doc_dict:
                doc_dict[k] = None
        
        documents.append(doc_dict)

    print(f"{program} - returning {len(documents)} documents, {skipped} were skipped due to missing necessary info.")
    
    return documents

Check that the function works correctly

In [9]:

audio_files = os.listdir(os.path.join(example_program_dir, audios_subdir))
text_files = os.listdir(os.path.join(example_program_dir, asr_subdir))

# Parse all documents from the example XML
all_documents = parse_docs_in_xml(xml_doc, example_program, text_files, audio_files)

print(f"Total documents found: {len(all_documents)}")
if all_documents:
    for idx, doc in enumerate(all_documents):
        #doc = all_documents[0]
        #print(f"\nFirst document keys: {list(doc.keys())}")
        print(f"\nTitle: {doc.get('title', 'N/A')[:80]}...")
        print(f"Broadcast program name: {doc.get('broadcast_program_name', 'N/A')}")
        print(f"Extracted boradcast date: {doc['date_str']} (year {doc['date_str'].split('/')[-1]})")
        print(f"Bradcast dates: {doc.get('first_broadcast_dates', ['N/A'])[0]}, Recording dates: {doc.get('recording_dates', ['N/A'])[0]}")
        print(f"MP3 filenames: {doc.get('mp3_filenames', ['N/A'])}")
        
        if doc['mp3_filenames']:
            for f in doc.get('mp3_filenames'):
                print(f" --> MP3 filename in audios: {f in audio_files}, OID in ASR files: {any(doc['stripped_OID'] in xml_f for xml_f in text_files)}")
        else:
            print(f"Doc {idx} has no mp3 filenames!! full doc:\n{doc}")
        #if doc.get('supports'):
        #    print(f"Audio file: {doc['supports'][0].get('filename', 'N/A')}")



Starting extracting 8 documents for program causerie_uni
Skipping document 3/8 with OID {1656BB3F-A8EF-4BF4-BE9A-20483F5665C8} because it's missing its XML file!! (stt exists: False)
Skipping document 5/8 with OID {87381313-87FD-4D31-AD19-8CDC08BA94DA} because it's missing its XML file!! (stt exists: False)
causerie_uni - returning 6 documents, 2 were skipped due to missing necessary info.
Total documents found: 6

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 19/12/1939 (year 1939)
Bradcast dates: 19/12/1939 - __/__/____, Recording dates: 20/11/1939 - 20/11/1939
MP3 filenames: ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3']
 --> MP3 filename in audios: True, OID in ASR files: True

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 07/01/1941 (year 1941)
Bradcast dates: 07/01/1941 - __/__/____, Recording dates: 26/11/1940 - 26/11/1940


In [10]:
all_documents[:5]

[{'alias': 'causerie_uni',
  'date_str': '19/12/1939',
  'mp3_filenames': ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3'],
  'cls_ID': '{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}',
  'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}',
  'stripped_OID': '3267F04D-657F-4DCB-BCB0-44B2B6C2682D',
  'login': '',
  'broadcast_episode_title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg",
  'sequence': '1',
  'broadcast_program_name': 'Causerie universitaire',
  'document_type': 'Parl?',
  'hierarchy_level': 'Sujet',
  'modified_by': 'albrecjo',
  'modified_on': '25.05.2022 03:56:53',
  'physical_support_history': 'Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B',
  'production_type': 'Production propre',
  'recording_place': 'Lausanne (Studio de Radio-Lausanne)',
  'rights_notes': 'Memoriav',
  'rights_status': 'Clarifi?',
  'series_title

In [171]:
doc_keys

dict_keys(['alias', 'date_str', 'mp3_filenames', 'cls_ID', 'OID', 'stripped_OID', 'login', 'broadcast_episode_title', 'sequence', 'broadcast_program_name', 'document_type', 'hierarchy_level', 'modified_by', 'modified_on', 'physical_support_history', 'production_type', 'recording_place', 'rights_notes', 'rights_status', 'series_title', 'content_summary', 'workflow_status', 'assembly_status', 'live', 'modilation_type', 'work_duration', 'work_duration_compl', 'participants', 'geographical_descriptors', 'person_descriptors', 'thematical_descriptors', 'rights_uage_possibilities', 'radio_channels', 'subdomains', 'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames'])

In [11]:
doc_keys = all_documents[0].keys()

out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/debug_metadata.csv"
with open(out_csv_name, "w", newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, doc_keys)
    dict_writer.writeheader()
    dict_writer.writerows(all_documents)


This works and yields the desired set of metadata. 

## Aggregating all the programs' metadata into a single file

Now we need to write an orchestrator function that opens and processes the XML documents for each provider, and stores the info in a dict or a dataframe, also extracting the first braodcast date to help with the creation of the issue index file

In [7]:
all_program_aliases = sorted([p for p in os.listdir(base_dir) if "DS_Store" not in p and "$RECYCLE" not in p and "System" not in p])
print(f"Found {len(all_program_aliases)} Radio programs: {all_program_aliases}")

Found 47 Radio programs: ['ana_media', 'bigbang', 'canal_euro', 'causerie_uni', 'chron_instit', 'chron_unesco', 'courrier_cr', 'culte', 'dos_sci', 'ecoute_paix', 'enquest', 'forum', 'forum_lau', 'geneve_info', 'hist_ondes', 'infopile', 'inst_monde', 'j13h', 'j13h2', 'j_mat', 'j_midi', 'j_nuit', 'j_soir', 'mag_eco', 'mag_info', 'mag_sci1', 'mag_sci2', 'mag_tv1', 'mag_tv2', 'mem_ondes', 'min_oecu', 'miroir_monde', 'miroir_temps', 'monde_ant', 'monde_sem', 'nickel', 'nu_parle', 'ombres_eco', 'paraboles', 'paris_parle', 'parole_prem', 'petitdej', 'suisse_euro', 'terre_ciel', 'trib_prem', 'vie_monde', 'vie_va']


In [8]:
out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts.csv"

In [184]:
all_program_aliases.index("mem_ondes")

29

In [29]:
already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
already_done = []

In [27]:
#already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
#already_done = []
already_done

[]

In [30]:
all_docs = []
all_docs_dict = {}
#doc_keys = None

for p_idx, p_alias in tqdm(enumerate(all_program_aliases)):

    if p_alias in already_done:
        print(f"\n{p_alias} is already done, skipping!")
        continue
    
    # first find all the metadata xml docs
    program_dir = os.path.join(base_dir, p_alias)
    metadata_files = [os.path.join(program_dir,f) for f in os.listdir(program_dir) if f.startswith(metadata_file_start)]
    
    all_audios_for_alias = os.listdir(os.path.join(program_dir,'audio'))
    all_stt_for_alias = os.listdir(os.path.join(program_dir,'stt'))

    print(f"\nPROCESSING PROGRAM {p_alias} ({p_idx+1}/{len(all_program_aliases)}) - {len(metadata_files)} files:")

    program_docs = []
    for xml_doc_path in tqdm(metadata_files):
        with open(xml_doc_path, "r", encoding="utf-8") as f:
            raw_xml = f.read()

        program_docs.extend(parse_docs_in_xml(BeautifulSoup(raw_xml, "xml"), p_alias, all_stt_for_alias, all_audios_for_alias, doc_keys))
    
    all_docs.extend(program_docs)
    all_docs_dict[p_alias] = program_docs

    # Save the current list of documents to save the progress
    #if not doc_keys:
    #    doc_keys = all_docs[0].keys()

    print(f" --> Adding {len(program_docs)} to the out csv for {p_alias}")
    with open(out_csv_name, "a", newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, doc_keys)
        if not already_done:
            dict_writer.writeheader()
        dict_writer.writerows(program_docs)

    already_done.append(p_alias)
        


0it [00:00, ?it/s]


PROCESSING PROGRAM ana_media (1/47) - 1 files:


100%|██████████| 1/1 [00:00<00:00, 15.58it/s]



Starting extracting 6 documents for program ana_media
ana_media - returning 6 documents, 0 were skipped due to missing necessary info.
 --> Adding 6 to the out csv for ana_media

PROCESSING PROGRAM bigbang (2/47) - 1 files:


100%|██████████| 1/1 [00:00<00:00,  3.70it/s]
2it [00:00,  5.57it/s]

...

100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
46it [02:47,  2.34s/it]

Skipping document 174/281 with OID {702CC5AB-8731-4188-AED1-686E0BEE217D} because it's missing its XML file!! (stt exists: False)
Skipping document 175/281 with OID {7A1F7137-B21C-4CB4-8ABE-3BE384EA0B2C} because it's missing its XML file!! (stt exists: False)
Skipping document 176/281 with OID {C084D9F6-7DF0-4EF2-A09C-E461C2CFB8A5} because it's missing its XML file!! (stt exists: False)
Skipping document 177/281 with OID {B126E6B2-F0FD-406D-9F58-8D055001D2C2} because it's missing its XML file!! (stt exists: False)
Skipping document 178/281 with OID {98167841-9284-4134-8ABB-FB82539D6507} because it's missing its XML file!! (stt exists: False)
Skipping document 179/281 with OID {5E43F5D7-78E1-4D45-BB46-5ED763B522BA} because it's missing its XML file!! (stt exists: False)
Skipping document 180/281 with OID {ACCA77DB-3930-4961-8B09-7E094FE9B8DB} because it's missing its XML file!! (stt exists: False)
Skipping document 182/281 with OID {1A917B8A-F511-41C4-8ECD-BC2776AB31CA} because it's mis


Starting extracting 295 documents for program vie_va
Skipping document 2/295 with OID {E77A33B9-F6E7-4110-A5FE-BE2BB93EBD22} because it's missing its XML file!! (stt exists: False)
Skipping document 3/295 with OID {9AC3A303-77E8-4127-9BE0-CC45935A7540} because it's missing its XML file!! (stt exists: False)
Skipping document 4/295 with OID {7B2B089A-C736-47E6-8BF0-63C8B8309363} because it's missing its XML file!! (stt exists: False)
Skipping document 5/295 with OID {FDE71322-57FA-4D9D-93FD-170155D065FC} because it's missing its XML file!! (stt exists: False)
Skipping document 6/295 with OID {1145C635-C7C5-4B28-94A7-AA61C72E4E78} because it's missing its XML file!! (stt exists: False)
Skipping document 7/295 with OID {A0C7D77E-205E-456F-ABC5-EEFAB87B8EC4} because it's missing its XML file!! (stt exists: False)
Skipping document 8/295 with OID {82FFEEF2-09FC-4B91-9C08-36595828A9DE} because it's missing its XML file!! (stt exists: False)
Skipping document 9/295 with OID {5633B243-30A9-4E

100%|██████████| 1/1 [00:01<00:00,  1.29s/it]
47it [02:49,  3.60s/it]

Skipping document 219/295 with OID {B8270A05-67CB-4DE4-AD0C-2C949FDEF755} because it's missing its XML file!! (stt exists: False)
Skipping document 263/295 with OID {8ABC766C-111F-4935-9112-74D993871D14} because it's missing its XML file!! (stt exists: False)
Skipping document 280/295 with OID {46E9ACB2-1871-4B2E-BE7F-6D019D99B0C6} because it's missing its XML file!! (stt exists: False)
Skipping document 281/295 with OID {D98C42AA-D98A-4180-A124-FF1B3CB42910} because it's missing its XML file!! (stt exists: False)
Skipping document 283/295 with OID {6D1F27BC-C83A-4A91-B10B-1F4EA5C8F01F} because it's missing its XML file!! (stt exists: False)
Skipping document 294/295 with OID {D978B28D-CAB3-4165-B3EE-77D973E36587} because it's missing its XML file!! (stt exists: False)
vie_va - returning 225 documents, 70 were skipped due to missing necessary info.
 --> Adding 225 to the out csv for vie_va


There were many duplicates due to a small mistake: the rows were all written again for each program

In [18]:
metadata_df = pd.read_csv(out_csv_name)
#metadata_df = metadata_df.drop_duplicates()
len(metadata_df)

26197

In [17]:
metadata_df.to_csv(out_csv_name, index=True)

In [10]:
v1_csv_name = out_csv_name.replace("rts", "rts_v1")
v1_csv_name

'/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts_v1.csv'

In [20]:
metadata_v1_df = pd.read_csv(v1_csv_name)
metadata_v1_df = metadata_v1_df.drop_duplicates()
len(metadata_v1_df)

36856

### Claude fix of the corrupted text where '?' replaces characters with accents

In [ ]:

# Now I understand all failures. Most of these are in MANUAL_MAP already.
# The rule failures are for tokens NOT in the manual map.
# Let me add the remaining edge cases to MANUAL_MAP and tighten the rules:
#
# Issues:
# 1. "St?r?o" - MANUAL_MAP has it, works ✓
# 2. "?lections" - MANUAL_MAP has it, works ✓ (same for all ?-initial words)
# 3. "Conf?rence" - MANUAL_MAP has it ✓
# 4. "m?me" - MANUAL_MAP has it ✓
# 5. "si?cle" - MANUAL_MAP has it ✓
# 6. "for?ts" - MANUAL_MAP has it ✓
# 7. "fa?on" - MANUAL_MAP has it ✓
# 8. "re?u" - MANUAL_MAP has it ✓
#
# ALL failures in the test are for tokens that ARE in the manual map!
# The test was using correct_by_rules directly without checking manual map first.
# The actual correct_token function checks manual map first, then falls back to rules.
# So the test was misleading.
#
# The ONLY critical rule fix is:
# 1. ç rule: '' in 'aou' = True -> need "after != '' and after.lower() in 'aou'"
# 2. à rule: fires before word-initial check - need to only fire when it's truly standalone
#    (not at position 0 of a word-token, since word-tokens don't start with spaces)
#    Actually, word-tokens extracted by re.split never have spaces, so before='' means i=0
#    Fix: only apply à rule if before=' ' (in context, not token-start)
#    Since we extract pure word tokens, before='' always means start of token -> should be é/è
#
# Let me write the final clean version and apply it.

import pandas as pd, re

# ─── FINAL MANUAL MAP (comprehensive) ───
MANUAL_MAP = {
    "?": "à",
    "Parl?": "Parlé", "'Parl?": "'Parlé", "Clarifi?": "Clarifié", "Valid?": "Validé",
    "Premi?re'": "Première'", "Accept?": "Accepté", "Mont?": "Monté",
    "Premi?re": "Première", "St?r?o": "Stéréo",
    "'Pr?sentateur": "'Présentateur", "Pr?sentateur": "Présentateur",
    "'Interview?": "'Interviewé", "Interview?": "Interviewé",
    "Gen?ve": "Genève", "'Gen?ve": "'Genève", "Gen?ve'": "Genève'",
    "Radio-Gen?ve": "Radio-Genève", "Radio-Gen?ve'": "Radio-Genève'",
    "Neuch?tel": "Neuchâtel", "'Neuch?tel": "'Neuchâtel", "Neuch?tel'": "Neuchâtel'",
    "neuch?telois": "neuchâtelois", "neuch?teloise": "neuchâteloise",
    "B?le": "Bâle", "Del?mont": "Delémont", "Z?rich": "Zürich",
    "Montr?al": "Montréal", "J?rusalem": "Jérusalem", "J?rusalem'": "Jérusalem'",
    "T?h?ran": "Téhéran", "Qu?bec": "Québec",
    "Alg?rie": "Algérie", "'Alg?rie'": "'Algérie'", "d'Alg?rie": "d'Algérie",
    "l'Alg?rie": "l'Algérie", "P?kin": "Pékin", "Cor?e": "Corée",
    "Ha?ti": "Haïti", "Za?re": "Zaïre", "Isra?l": "Israël",
    "'Isra?l'": "'Israël'", "d'Isra?l": "d'Israël", "Gr?ce": "Grèce",
    "Su?de": "Suède", "Kowe?t": "Koweït", "Am?rique": "Amérique",
    "Bosnie-Herz?govine": "Bosnie-Herzégovine",
    "'Bosnie-Herz?govine'": "'Bosnie-Herzégovine'", "Herz?govine": "Herzégovine",
    "Andr?": "André", "Andr?'": "André'", "d'Andr?": "d'André",
    "Ren?": "René", "Ren?'": "René'", "Jean-Ren?'": "Jean-René'",
    "Fran?ois": "François", "Fran?ois'": "François'",
    "Jean-Fran?ois": "Jean-François", "Jean-Fran?ois'": "Jean-François'",
    "Fran?oise": "Françoise", "Fran?oise'": "Françoise'",
    "G?rard": "Gérard", "G?rard'": "Gérard'",
    "Rapha?l": "Raphaël", "Rapha?l'": "Raphaël'",
    "Jos?": "José", "Mich?le": "Michèle", "Mich?le'": "Michèle'",
    "Marie-Jos?'": "Marie-José'", "L?on": "Léon", "L?o": "Léo",
    "Fr?d?ric": "Frédéric", "Fr?d?ric'": "Frédéric'",
    "G?rald": "Gérald", "F?lix": "Félix", "F?licien": "Félicien",
    "Mikha?l": "Mikhaïl", "J?rg": "Jörg", "J?rg'": "Jörg'",
    "Val?ry": "Valéry", "R?my": "Rémy", "G?n?ral": "Général",
    "No?l": "Noël", "Genevi?ve": "Geneviève", "B?at'": "Béat'",
    "C?cile'": "Cécile'", "Marl?ne'": "Marlène'",
    "Pierre-Andr?": "Pierre-André", "Pierre-Andr?'": "Pierre-André'",
    "Georges-Andr?": "Georges-André", "Georges-Andr?'": "Georges-André'",
    "St?phane": "Stéphane", "St?phane'": "Stéphane'",
    "Fran?ais": "Français", "Ma?tre": "Maître",
    "B?guin": "Béguin", "'B?guin": "'Béguin", "B?guelin": "Béguelin", "'B?guelin": "'Béguelin",
    "Fran?ois-Achille'": "François-Achille'",
    "Nussl?": "Nusslé", "'Nussl?": "'Nusslé", "Werl?": "Werlé", "'Werl?": "'Werlé",
    "K?nzler": "Künzler", "'K?nzler": "'Künzler",
    "M?ller": "Müller", "'M?ller": "'Müller",
    "Gilli?ron": "Gilliéron", "'Gilli?ron": "'Gilliéron",
    "Moss?": "Mossé", "'Moss?": "'Mossé",
    "D?caillet": "Décaillet", "'D?caillet": "'Décaillet",
    "D?cotte": "Décotte", "'D?cotte": "'Décotte",
    "pr?sident": "président", "'pr?sident": "'président",
    "vice-pr?sident": "vice-président", "'vice-pr?sident": "'vice-président",
    "Pr?sident": "Président", "Pr?sentation": "Présentation",
    "pr?sidente": "présidente", "pr?sidentielle": "présidentielle",
    "pr?sidentielle'": "présidentielle'", "pr?sidentielles": "présidentielles",
    "pr?sidence": "présidence", "pr?sent": "présent",
    "pr?sent?": "présenté", "pr?sente": "présente", "pr?sentation": "présentation",
    "f?d?ral": "fédéral", "f?d?ral'": "fédéral'", "f?d?rale": "fédérale",
    "f?d?rale'": "fédérale'", "f?d?rales": "fédérales",
    "f?d?ration": "fédération", "F?d?ration": "Fédération", "F?d?ral": "Fédéral",
    "Conf?d?ration": "Confédération", "Conf?d?ration'": "Confédération'",
    "fran?ais": "français", "fran?ais'": "français'",
    "fran?aise": "française", "fran?aise'": "française'", "fran?aises": "françaises",
    "d?claration": "déclaration", "D?claration": "Déclaration",
    "'D?claration": "'Déclaration", "'D?claration'": "'Déclaration'",
    "d?clarations": "déclarations",
    "apr?s": "après", "Apr?s": "Après", "Apr?s'": "Après'", "aupr?s": "auprès",
    "g?n?ral": "général", "g?n?rale": "générale",
    "?t?": "été", "?crivain": "écrivain", "'?crivain'": "'écrivain'",
    "'?crivain": "'écrivain", "?crivain'": "écrivain'", "l'?crivain": "l'écrivain",
    "?crivaine": "écrivaine", "'?crivaine'": "'écrivaine'", "?crivaine'": "écrivaine'",
    "secr?taire": "secrétaire", "'secr?taire": "'secrétaire",
    "D?partement": "Département", "d?partement": "département",
    "?me": "ème", "Conf?rence": "Conférence", "'Conf?rence": "'Conférence",
    "conf?rence": "conférence", "sp?cial": "spécial", "sp?ciale": "spéciale",
    "sp?cialiste": "spécialiste", "l'Universit?": "l'Université",
    "l'universit?": "l'université",
    "?conomique": "économique", "?conomique'": "économique'",
    "?conomiques": "économiques", "?conomie": "économie",
    "l'?conomie": "l'économie", "'?conomie'": "'économie'",
    "d'?conomie": "d'économie", "?conomiste": "économiste", "?conomiste'": "économiste'",
    "R?action": "Réaction", "r?action": "réaction", "r?actions": "réactions",
    "R?actions": "Réactions", "R?publique": "République", "r?publique": "république",
    "T?moignage": "Témoignage", "t?moignage": "témoignage",
    "t?moignages": "témoignages", "T?moignages": "Témoignages", "t?moin": "témoin",
    "?tre": "être", "d'?tre": "d'être", "Pr?t": "Prêt", "pr?t": "prêt",
    "pr?ts": "prêts", "pr?tre": "prêtre", "'pr?tre'": "'prêtre'", "pr?tres": "prêtres",
    "?missions": "émissions", "l'?mission": "l'émission", "'?mission": "'émission",
    "?mission": "émission",
    "premi?re": "première", "premi?res": "premières",
    "oecum?nique": "œcuménique", "'oecum?nisme'": "'œcuménisme'",
    "sovi?tique": "soviétique", "sovi?tiques": "soviétiques", "Sovi?tiques": "Soviétiques",
    "am?ricain": "américain", "am?ricaine": "américaine",
    "am?ricains": "américains", "am?ricaines": "américaines", "Am?ricains": "Américains",
    "l'arm?e": "l'armée", "'arm?e'": "'armée'", "arm?e": "armée", "arm?'": "armé'",
    "soci?t?": "société", "Soci?t?": "Société", "soci?t?s": "sociétés",
    "'Deuxi?me": "'Deuxième", "Deuxi?me": "Deuxième", "deuxi?me": "deuxième",
    "r?fugi?s": "réfugiés", "r?fugi?": "réfugié", "r?fugi?'": "réfugié'",
    "'r?fugi?'": "'réfugié'", "r?fugi?e": "réfugiée",
    "?lections": "élections", "'?lection": "'élection", "?lection": "élection",
    "'?lection'": "'élection'", "l'?lection": "l'élection", "?lectorale": "électorale",
    "isra?lien": "israélien", "isra?lienne": "israélienne",
    "r?gion": "région", "r?gions": "régions", "n?gociations": "négociations",
    "c?r?monie": "cérémonie", "C?r?monie": "Cérémonie",
    "m?me": "même", "m?mes": "mêmes", "sc?ne": "scène",
    "Comit?": "Comité", "comit?": "comité",
    "g?n?rale": "générale", "com?dien": "comédien",
    "'com?dien'": "'comédien'", "'com?dien": "'comédien",
    "com?dienne": "comédienne", "'com?dienne'": "'comédienne'",
    "Congr?s": "Congrès", "congr?s": "congrès",
    "l'ann?e": "l'année", "ann?e": "année", "ann?es": "années",
    "Enqu?te": "Enquête", "d'enqu?te": "d'enquête", "enqu?te": "enquête",
    "l'enqu?te": "l'enquête", "d?part": "départ", "march?": "marché",
    "March?": "Marché", "d?mission": "démission", "D?mission": "Démission",
    "d?but": "début", "D?but": "Début", "charg?": "chargé", "d?fense": "défense",
    "gr?ve": "grève", "'gr?ve'": "'grève'",
    "l'Assembl?e": "l'Assemblée", "l'assembl?e": "l'assemblée",
    "communaut?": "communauté", "Communaut?": "Communauté", "r?gime": "régime",
    "?lu": "élu", "?lus": "élus", "?lue": "élue",
    "d?veloppement": "développement", "po?te": "poète",
    "autorit?s": "autorités", "cons?quences": "conséquences", "libert?": "liberté",
    "th??tre": "théâtre", "Th??tre": "Théâtre",
    "'cin?ma'": "'cinéma'", "cin?ma": "cinéma",
    "m?re": "mère", "P?re": "Père", "p?re": "père",
    "fr?re": "frère", "fr?res": "frères",
    "F?te": "Fête", "f?te": "fête", "'f?te": "'fête", "f?tes": "fêtes",
    "f?vrier": "février",
    "?tat": "État", "d'?tat": "d'État", "l'?tat": "l'État", "?tats": "États",
    "t?te": "tête", "bless?s": "blessés", "r?alisateur": "réalisateur",
    "d?bat": "débat", "D?bat": "Débat", "'D?bat'": "'Débat'",
    "d?bat'": "débat'", "D?bat'": "Débat'",
    "ch?mage": "chômage", "'ch?mage'": "'chômage'", "ch?meurs": "chômeurs",
    "pr?sence": "présence", "pass?": "passé", "journ?e": "journée", "Journ?e": "Journée",
    "l'?tranger": "l'étranger", "ing?nieur": "ingénieur", "d?s": "dès",
    "?ge": "âge", "l'?ge": "l'âge",
    "?galement": "également", "?v?nements": "événements", "'?v?nement": "'événement",
    "difficult?s": "difficultés", "D?c?s": "Décès", "d?c?s": "décès",
    "d?cembre": "décembre",
    "l'?cole": "l'école", "?cole": "école", "'?coles": "'écoles", "?coles": "écoles",
    "re?u": "reçu", "contr?le": "contrôle",
    "r?union": "réunion", "R?union": "Réunion", "'r?union": "'réunion",
    "solidarit?": "solidarité", "Solidarit?": "Solidarité",
    "si?ge": "siège", "l'a?roport": "l'aéroport",
    "a?rienne": "aérienne", "a?rienne'": "aérienne'", "a?rien'": "aérien'",
    "a?rostier": "aérostier", "ao?t": "août",
    "cha?ne": "chaîne", "Cha?ne": "Chaîne",
    "condamn?": "condamné", "r?sistance": "résistance",
    "?tudiants": "étudiants", "?tudiant": "étudiant",
    "r?forme": "réforme", "r?formes": "réformes", "d?mocratie": "démocratie",
    "'d?mocratie": "'démocratie", "carri?re": "carrière",
    "ext?rieur": "extérieur", "ext?rieure": "extérieure",
    "l?gislatives": "législatives", "'l?gislation'": "'législation'",
    "'l?gislatif": "'législatif",
    "?v?que": "évêque", "?v?ques": "évêques",
    "?trangers": "étrangers", "?trang?res": "étrangères",
    "?trang?re": "étrangère", "?trang?res'": "étrangères'",
    "fronti?res": "frontières", "fronti?re": "frontière",
    "r?f?rendum": "référendum", "'r?f?rendum'": "'référendum'",
    "?voque": "évoque", "'ex?cutif": "'exécutif",
    "Mus?e": "Musée", "mus?e": "musée", "pi?ce": "pièce",
    "Pr?dication": "Prédication", "pr?dication": "prédication",
    "r?volution": "révolution", "litt?rature": "littérature",
    "'litt?rature'": "'littérature'", "litt?raire": "littéraire",
    "litt?raire'": "littéraire'",
    "l'ind?pendance": "l'indépendance", "'ind?pendantisme'": "'indépendantisme'",
    "r?daction": "rédaction", "mati?re": "matière", "donn?": "donné",
    "d?cid?": "décidé", "syst?me": "système", "succ?s": "succès",
    "r?vision": "révision", "d?couverte": "découverte",
    "arr?t?": "arrêté", "Arr?t": "Arrêt",
    "financi?re": "financière", "sign?": "signé",
    "m?decine": "médecine", "'m?decine'": "'médecine'",
    "m?decin": "médecin", "'m?decin": "'médecin", "'m?decin'": "'médecin'",
    "m?decins": "médecins", "M?decins": "Médecins",
    "T?l?vision": "Télévision", "'t?l?vision'": "'télévision'",
    "t?l?vision": "télévision", "t?l?phone": "téléphone",
    "derni?re": "dernière", "derni?res": "dernières",
    "l'int?rieur": "l'intérieur", "l'Int?rieur": "l'Intérieur",
    "l'arriv?e": "l'arrivée", "arriv?e": "arrivée", "Arriv?e": "Arrivée",
    "?taient": "étaient", "?tait": "était",
    "M?moire": "Mémoire", "m?moire": "mémoire",
    "Lib?ration": "Libération", "lib?ration": "libération",
    "lib?ral": "libéral", "lib?rale": "libérale",
    "cr?ation": "création", "Cr?ation": "Création",
    "envoy?": "envoyé", "'envoy?": "'envoyé",
    "d?l?gation": "délégation", "d?l?gu?": "délégué",
    "'d?l?gu?": "'délégué", "d?l?gu?s": "délégués", "'d?l?gu?s": "'délégués",
    "r?le": "rôle", "r?les": "rôles",
    "s?curit?": "sécurité", "proc?s": "procès", "'proc?s'": "'procès'",
    "r?dacteur": "rédacteur", "'r?dacteur": "'rédacteur",
    "si?cle": "siècle", "si?cle'": "siècle'",
    "d?put?": "député", "'d?put?": "'député", "d?put?e": "députée",
    "d?put?s": "députés", "d?put?es": "députées",
    "d?cision": "décision", "D?cision": "Décision",
    "l'?nergie": "l'énergie", "'?nergie": "'énergie", "?nergies": "énergies",
    "repr?sentant": "représentant", "repr?sentants": "représentants",
    "conseill?re": "conseillère", "'conseill?re": "'conseillère",
    "d'?tat": "d'État", "pr?s": "près",
    "comp?tition": "compétition", "sant?": "santé",
    "R?trospective": "Rétrospective", "c?t?": "côté", "m?dias": "médias",
    "l'adh?sion": "l'adhésion", "r?sultats": "résultats", "r?sultat": "résultat",
    "Europ?enne": "Européenne", "europ?enne": "européenne",
    "europ?enne'": "européenne'", "europ?ennes": "européennes",
    "europ?en": "européen", "europ?ens": "européens",
    "malgr?": "malgré", "coop?ration": "coopération",
    "th?ologie": "théologie", "th?ologien": "théologien",
    "alg?rien": "algérien", "minist?re": "ministère",
    "d?": "dé", "d?mocrate-chr?tien": "démocrate-chrétien",
    "pr?nom": "prénom", "chr?tiens": "chrétiens", "chr?tien": "chrétien",
    "th?me": "thème", "employ?s": "employés",
    "f?d?ration": "fédération", "organis?": "organisé",
    "S?rie": "Série", "s?rie": "série",
    "diff?rents": "différents", "diff?rentes": "différentes",
    "diff?rent": "différent", "m?tier": "métier",
    "entra?neur": "entraîneur", "laur?at": "lauréat",
    "consacr?": "consacré", "consacr?e": "consacrée",
    "?tudes": "études", "d'?tudes": "d'études", "?tude": "étude",
    "?pouse": "épouse", "r?ponse": "réponse",
    "n?cessit?": "nécessité", "?meutes": "émeutes",
    "l'?le": "l'île", "for?ts": "forêts",
    "m?dicale": "médicale", "responsabilit?": "responsabilité",
    "personnalit?s": "personnalités", "personnalit?": "personnalité",
    "op?ration": "opération", "Op?ration": "Opération", "op?rations": "opérations",
    "proc?dure": "procédure", "d?tenus": "détenus", "trouv?": "trouvé",
    "pr?vention": "prévention", "publicit?": "publicité",
    "neutralit?": "neutralité", "r?unification": "réunification",
    "nucl?aire": "nucléaire", "nucl?aire'": "nucléaire'", "nucl?aires": "nucléaires",
    "l'?quipe": "l'équipe", "?change": "échange", "o?": "où", "tr?s": "très",
    "probl?me": "problème", "probl?mes": "problèmes",
    "nomm?": "nommé", "strat?gie": "stratégie",
    "activit?": "activité", "activit?s": "activités",
    "exp?rience": "expérience", "propri?taire": "propriétaire",
    "Nestl?": "Nestlé", "annonc?": "annoncé", "priv?e": "privée",
    "col?re": "colère", "accept?": "accepté",
    "Tch?coslovaquie": "Tchécoslovaquie", "rencontr?": "rencontré",
    "Di?te": "Diète", "lanc?": "lancé", "cr?er": "créer",
    "Proc?s": "Procès", "m?daille": "médaille",
    "p?dophile": "pédophile", "horlog?re": "horlogère",
    "fa?on": "façon", "reconna?t": "reconnaît",
    "al?manique": "alémanique", "v?cu": "vécu",
    "'comm?moration'": "'commémoration'", "p?trole": "pétrole",
    "lumi?res": "lumières", "l'abb?": "l'abbé", "l'h?pital": "l'hôpital",
    "mani?re": "manière", "helv?tique": "helvétique",
    "d?mocrate": "démocrate", "?l?ments": "éléments",
    "d?nonce": "dénonce", "d?faite": "défaite", "plut?t": "plutôt",
    "gr?ce": "grâce", "volont?": "volonté", "b?timent": "bâtiment",
    "l'op?ration": "l'opération", "stup?fiants'": "stupéfiants'",
    "jusqu'?": "jusqu'à",
    "requ?rant": "requérant", "requ?rants": "requérants",
    "assassin?": "assassiné", "r?alit?": "réalité",
    "l'Acad?mie": "l'Académie", "d?cide": "décide",
    "cin?aste": "cinéaste", "d?tention": "détention",
    "l'?volution": "l'évolution", "l'?gard": "l'égard",
    "?glise": "église", "l'?glise": "l'église", "?glises": "églises",
    "'?glise": "'église",
    "R?cit": "Récit", "r?cit": "récit", "'R?citant": "'Récitant",
    "?crit": "écrit", "?crits": "écrits", "?crite": "écrite",
    "f?minin": "féminin", "occup?s": "occupés",
    "majorit?": "majorité", "?tabli": "établi", "?tablie": "établie",
    "?le": "île", "?le'": "île'",
    "?lecteur": "électeur", "?lecteurs": "électeurs",
    "?lection": "élection", "?lection'": "élection'",
    "?lectricit?": "électricité",
    "?levage": "élevage", "?leveur": "éleveur", "?leveurs": "éleveurs",
    "?lite": "élite", "?loge": "éloge",
    "?boueur": "éboueur", "?branlement": "ébranlement",
    "?cart": "écart", "?chafaud": "échafaud",
    "?chec": "échec", "?checs'": "échecs'",
    "?chographie'": "échographie'", "?clairage": "éclairage",
    "?clipse'": "éclipse'", "?coli?re": "écolière",
    "?colier": "écolier", "?colier'": "écolier'", "?coliers": "écoliers",
    "?cologie": "écologie", "?cologie'": "écologie'", "?cologisme'": "écologisme'",
    "?conomie": "économie", "?conomie'": "économie'", "?conomies": "économies",
    "?coute": "écoute", "?craser": "écraser", "?crevisse'": "écrevisse'",
    "?criture": "écriture", "?criture'": "écriture'",
    "?crivain": "écrivain", "?crivains": "écrivains",
    "?dition": "édition", "?dition'": "édition'",
    "?ducateur": "éducateur", "?ducateur'": "éducateur'",
    "?ducateurs": "éducateurs", "?ducation": "éducation", "?ducation'": "éducation'",
    "?ducatrice": "éducatrice",
    "?galit?": "égalité", "?galit?'": "égalité'",
    "?gyptologue": "égyptologue", "?gyptologue'": "égyptologue'",
    "?largir": "élargir",
    "?l?ve'": "élève'", "?l?ves": "élèves", "?l?phant'": "éléphant'",
    "?cologiste": "écologiste", "?cologiste'": "écologiste'",
    "?ll?s": "elles",
    "d?j?": "déjà",
    "carr?": "carré",
    "t?l?vis?": "télévisé",
    "f?d?r?": "fédéré",
    "diff?renci?": "différencié",
    "?v?nement": "événement", "?v?nements": "événements",
    "?v?que": "évêque",
    "soci?t?": "société",
    "t?moignage": "témoignage",
    "repr?senter": "représenter",
    "s?curit?s": "sécurités",
    "r?sistance": "résistance",
    "th?orie": "théorie",
    "cin?matographique": "cinématographique",
    "po?sie": "poésie",
    "'po?sie'": "'poésie'",
    "int?gration": "intégration",
    "'int?gration": "'intégration",
    "l'int?gration": "l'intégration",
    "d?l?gu?s": "délégués",
    "'d?put?": "'député",
    "d?bats": "débats",
    "repr?sente": "représente",
    "abb?": "abbé",
    "t?l?": "télé",
    "?crivains": "écrivains",
    "?dition": "édition",
    "p?re": "père",
    "carr?": "carré",
    "arm?": "armé",
    "g?ographe": "géographe",
    "g?ographie": "géographie",
    "g?ographique": "géographique",
    "arch?ologue": "archéologue",
    "arch?ologie": "archéologie",
    "Universit?s": "Universités",
    "universit?s": "universités",
    "M?t?o": "Météo",
    "li?": "lié",
    "L'?volution": "L'évolution",
    "l'?ch?ance": "l'échéance",
    "adh?sion": "adhésion",
    "El?ment": "Elément",
    "el?ments": "eléments",
    "install?s": "installés",
    "chass?s": "chassés",
    "marqu?s": "marqués",
    "marqu?": "marqué",
    "cha?nes": "chaînes",
    "l'entra?neur": "l'entraîneur",
    "au-del?": "au-delà",
    "Jusqu'o?": "Jusqu'où",
    "men?s": "menés",
    "Plan?te": "Planète",
    "concr?tement": "concrètement",
    "d?veloppment": "développement",
    "J?r?me": "Jérôme",
    "requ?tes": "requêtes",
    # all the words ending in -s to remove the rule which broke many participe passés
    "proc?s": "procès",
    "apr?s": "après",
    "tr?s": "très",
    "succ?s": "succès",
    "exc?s": "excès",
    "acc?s": "accès",
    "progr?s": "progrès",
    "congr?s": "congrès",
    "d?c?s": "décès",
    "expr?s": "exprès",
    "aupr?s": "auprès",
    "pr?s": "près",
    "d?s": "dès",
}

# Suffixes after which a word-final ? is a real question mark, not an accent
_REAL_QUESTION_SUFFIXES = (
    # verb infinitives
    'er', 'ir', 're', 'dre', 'tre', 'ndre', 'indre', 'oudre',
    # common noun/adj endings that don't take accents
    'eur', 'eur', 'oir', 'our', 'eur', "istes",
    # explicit vowel endings
    'a', 'e', 'i', 'o', 'u', 'y',
)


def _is_real_question_mark(token, pos):
    """Return True if the ? at position pos in token is a genuine question mark."""
    # Only applies to word-final ?
    if pos != len(token) - 1:
        return False
    before = token[:pos].lower()
    # After a vowel directly
    if before and before[-1] in 'aeiouy':
        return True
    # After known suffixes that never take a final accent
    for suffix in _REAL_QUESTION_SUFFIXES:
        if before.endswith(suffix):
            return True
    return False

def correct_by_rules(token, char='?'):
    if char not in token:
        return token
    result = list(token)
    i = 0
    while i < len(result):
        if result[i] != char:
            i += 1
            continue
        before = result[i-1] if i > 0 else ''
        after = result[i+1] if i < len(result)-1 else ''
        after2 = result[i+2] if i < len(result)-2 else ''

        # ── Real question mark? (word-final after verb/noun endings) ──
        if char=='?' and _is_real_question_mark(token, i):
            result[i] = '?'

        # ç: ? before a/o/u after consonant (after MUST be non-empty)
        elif after and after.lower() in 'aou' and before.lower() in 'cfglmnst':
            # Restrict ç: mainly after n, c, l, s, g, f (not r which is rare)
            result[i] = 'ç'
        # è before -re at non-word-final position (ère endings)
        # Only if followed by 're' and that's near end of word
        elif after.lower() == 'r' and after2.lower() == 'e' and (i+3 >= len(result) or result[i+3] in ('s', '', 'z')):
            result[i] = 'è'
        # è before -me (problème, même) - ê for même is in manual map
        elif after.lower() == 'm' and after2.lower() == 'e':
            result[i] = 'è'
        # è before -ve (grève) - in manual map
        elif after.lower() == 'v' and after2.lower() == 'e':
            result[i] = 'è'
        # è before -ge (siège) - in manual map
        elif after.lower() == 'g' and after2.lower() == 'e':
            result[i] = 'è'
        # è before final -s (procès, après, très)
        #elif after.lower() == 's' and (i+2 >= len(result) or result[i+2] in (' ', '', "'", ',', '.', ')', ']', '"', '-')):
        #    result[i] = 'è'
        # è before -ce (pièce)
        elif after.lower() == 'c' and after2.lower() == 'e':
            result[i] = 'è'
        # è: i?r pattern (première)
        elif before.lower() == 'i' and after.lower() == 'r':
            result[i] = 'è'
        # è: ?v (Genève: n?v → è)
        elif after.lower() == 'v':
            result[i] = 'è'
        # ê before final -t (arrêt, forêt) - in manual map
        elif after.lower() == 't' and (i+2 >= len(result) or result[i+2] in (' ', '', "'", ',', '.', ')')):
            result[i] = 'ê'
        # ô after h
        elif before.lower() == 'h':
            result[i] = 'ô'
        # Word-initial (i=0 in token): ? → mostly é
        elif i == 0:
            if after.lower() == 'l' and after2 and after2.lower() not in 'aeioué':
                result[i] = 'î'   # île
            elif after.lower() == 'g' and after2.lower() == 'e':
                result[i] = 'â'   # âge
            else:
                result[i] = 'é'
        # Word-final ? after vowel 'e' → likely real question mark
        elif not after and before.lower() in 'aeiouy':
            result[i] = '?'   # Keep as real ?
        # Default: é
        else:
            result[i] = 'é'
        i += 1
    return ''.join(result)

def correct_token(token, char="?", _map = MANUAL_MAP):
    if token in _map:
        return _map[token]
    return correct_by_rules(token, char=char)

def fix_cell(cell_value, char="?", _map=MANUAL_MAP):
    if not isinstance(cell_value, str) or char not in cell_value:
        return cell_value
    if char == "?":
        parts = re.split(r"([A-Za-z?àâäéèêëîïôùûüçœæ'-]+)", cell_value)
    elif char == "�":
        parts = re.split(r"([A-Za-z�àâäéèêëîïôùûüçœæ'-]+)", cell_value)
    result = []
    for part in parts:
        if char in part and re.search(r'[A-Za-z]', part):
            result.append(correct_token(part, char=char, _map=_map))
        elif part == char:
            result.append('à')   # standalone ? is always "à"
        else:
            result.append(part)
    return ''.join(result)

ID_COLS = {
    'alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID',
    'exact_date', 'broadcast_date', 'cls_ID', 'OID', 'login',
    'sequence', 'modified_by', 'modified_on', 'work_duration',
    'work_duration_compl', 'spt_filenames'
}

#### Try to repreat the approach with the character "�"

Claude's approach fixed the problems with the character "?" but another character is there.

However there are still quite a bit of problems with the fix. I found more examples which had erroneous corrections but much more probably exist.
What we did was the best we could.

In [48]:
SECOND_MANUAL_MAP = {k.replace("?", "�"): v for k,v in MANUAL_MAP.items()}
# add examples by hand from errors remaining spotted
SECOND_MANUAL_MAP["Ch�teaureynaud"] = "Châteaureynaud"
SECOND_MANUAL_MAP

{'�': 'à',
 'Parl�': 'Parlé',
 "'Parl�": "'Parlé",
 'Clarifi�': 'Clarifié',
 'Valid�': 'Validé',
 "Premi�re'": "Première'",
 'Accept�': 'Accepté',
 'Mont�': 'Monté',
 'Premi�re': 'Première',
 'St�r�o': 'Stéréo',
 "'Pr�sentateur": "'Présentateur",
 'Pr�sentateur': 'Présentateur',
 "'Interview�": "'Interviewé",
 'Interview�': 'Interviewé',
 'Gen�ve': 'Genève',
 "'Gen�ve": "'Genève",
 "Gen�ve'": "Genève'",
 'Radio-Gen�ve': 'Radio-Genève',
 "Radio-Gen�ve'": "Radio-Genève'",
 'Neuch�tel': 'Neuchâtel',
 "'Neuch�tel": "'Neuchâtel",
 "Neuch�tel'": "Neuchâtel'",
 'neuch�telois': 'neuchâtelois',
 'neuch�teloise': 'neuchâteloise',
 'B�le': 'Bâle',
 'Del�mont': 'Delémont',
 'Z�rich': 'Zürich',
 'Montr�al': 'Montréal',
 'J�rusalem': 'Jérusalem',
 "J�rusalem'": "Jérusalem'",
 'T�h�ran': 'Téhéran',
 'Qu�bec': 'Québec',
 'Alg�rie': 'Algérie',
 "'Alg�rie'": "'Algérie'",
 "d'Alg�rie": "d'Algérie",
 "l'Alg�rie": "l'Algérie",
 'P�kin': 'Pékin',
 'Cor�e': 'Corée',
 'Ha�ti': 'Haïti',
 'Za�re': 'Zaïre',
 'Is

In [51]:
df_to_fix_path = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts_v3.csv"
df_fixed_path_int = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts_fixed_int.csv"
df_fixed_path = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts_fixed.csv"

df_to_fix = pd.read_csv(df_to_fix_path)

TEXT_COLS = [c for c in df_to_fix.columns if c not in ID_COLS]

print("Processing cells for char = '?' ...")
total_fixes = 0
for col in TEXT_COLS:
    before = df_to_fix[col].copy()
    df_to_fix[col] = df_to_fix[col].apply(fix_cell)
    fixes = (before != df_to_fix[col]).sum()
    total_fixes += fixes
    if fixes > 0:
        print(f"  {col}: {fixes} cells modified")

print(f"\nTotal cells modified: {total_fixes}")

# Verify key rows
print("\n--- Sample verification ---")
check_cols = ['broadcast_episode_title', 'content_summary', 'workflow_status', 'document_type', 'modulation_type']
for col in check_cols:
    samples = df_to_fix[col].dropna().head(5).tolist()
    print(f"\n{col}:")
    for s in samples[:3]:
        print(f"  {str(s)[:120]}")


print("\nSaving...")
df_to_fix.to_csv(df_fixed_path_int, index=False)


print("Processing cells for char = '�' ...")
total_fixes = 0
for col in TEXT_COLS:
    before = df_to_fix[col].copy()
    df_to_fix[col] = df_to_fix[col].apply(fix_cell, char='�', _map=SECOND_MANUAL_MAP)
    fixes = (before != df_to_fix[col]).sum()
    total_fixes += fixes
    if fixes > 0:
        print(f"  {col}: {fixes} cells modified")

print(f"\nTotal cells modified: {total_fixes}")

# Verify key rows
print("\n--- Sample verification ---")
check_cols = ['broadcast_episode_title', 'content_summary', 'workflow_status', 'document_type', 'modulation_type']
for col in check_cols:
    samples = df_to_fix[col].dropna().head(5).tolist()
    print(f"\n{col}:")
    for s in samples[:3]:
        print(f"  {str(s)[:120]}")

print("\nSaving...")
df_to_fix.to_csv(df_fixed_path, index=False)
print("Done!")

Processing cells for char = '?' ...
  broadcast_episode_title: 19134 cells modified
  broadcast_program_name: 1549 cells modified
  document_type: 20956 cells modified
  physical_support_history: 3576 cells modified
  production_type: 521 cells modified
  recording_place: 19605 cells modified
  rights_notes: 22818 cells modified
  rights_status: 22332 cells modified
  series_title: 8917 cells modified
  content_summary: 21474 cells modified
  workflow_status: 20956 cells modified
  assembly_status: 22585 cells modified
  live: 670 cells modified
  modulation_type: 4832 cells modified
  geographical_descriptors: 15692 cells modified
  person_descriptors: 16951 cells modified
  thematical_descriptors: 18174 cells modified
  rights_usage_possibilities: 8538 cells modified
  radio_channels: 15075 cells modified
  subdomains: 10830 cells modified
  recording_dates: 492 cells modified
  first_broadcast_dates: 1056 cells modified
  supports: 16948 cells modified
  participants: 17654 cells mo

## Perform some additional post-processing and cleaning 

This is specifically to separate the case for the metadata and issue index creation

In [ ]:
rts_metadata_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts.csv"

rts_metadata_df = pd.read_csv(rts_metadata_filepath)
print(len(rts_metadata_df))
rts_metadata_df.head()

26261


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,person_descriptors,thematical_descriptors,rights_usage_possibilities,radio_channels,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},NaN,...,"['X-Files', 'Aux frontières du réel']","['série télèvisée', 'science-fiction']",NaN,['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '7UBM_057338_1{8cfcf300-a815-4f92-bea2-...","[{'name': 'Frias, Roxanne', 'function': 'Inter..."
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},NaN,...,NaN,"['école primaire', 'Internet', 'technique péda...",NaN,['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['1BBM_058066_6{2782ba11-7050-44d3-bef1-a81252...,"[{'name': 'Dubois, Laurent', 'function': 'Inte..."
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{2AF13E50-C6A2-49E5-9851-06C441833B50},NaN,...,"['SSR', 'TSR']","['censure', 'télévision', 'morale']",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'RPBM_057474_1{8116d0d3-1c12-44cf-a8a2-...","[{'name': 'Duparc, Nicole', 'function': 'Inter..."
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{C940BD19-5503-4918-852B-D40C1C18B359},NaN,...,NaN,"['série télèvisée', 'urgence médicale', 'télés...",NaN,['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['O1BM_057153_2{396b576c-7ed9-4018-a251-c12a84...,"[{'name': 'Kiefer, B.', 'function': 'Interview..."
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},NaN,...,NaN,"['enseignement secondaire', 'technique pédagog...",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['FHBM_055791_2{e8455981-078d-44f2-bb94-2e22cd...,"[{'name': 'Prophan, Geneviève', 'function': 'I..."


#### Columns filtering 

There was originally 36856 documents in the metadata, 10659 of which were missing MP3 and/or XML files (26261 had both). 

First remove the unused columns in this context, as well as all broadcasts which have no dates.

There are two main uses: the metadata, and the issue index. The filtering for the issue index columns will take place later on

In [5]:
rts_metadata_df.columns

Index(['alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID',
       'exact_date', 'broadcast_date', 'cls_ID', 'OID', 'login',
       'broadcast_episode_title', 'sequence', 'broadcast_program_name',
       'document_type', 'hierarchy_level', 'modified_by', 'modified_on',
       'physical_support_history', 'production_type', 'recording_place',
       'rights_notes', 'rights_status', 'series_title', 'content_summary',
       'workflow_status', 'assembly_status', 'live', 'modulation_type',
       'work_duration', 'work_duration_compl', 'geographical_descriptors',
       'person_descriptors', 'thematical_descriptors',
       'rights_usage_possibilities', 'radio_channels', 'subdomains',
       'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames',
       'participants'],
      dtype='str')

In [6]:
# remove all the broadcasts without a date as we cannot process them
valid_rts_df = rts_metadata_df[~rts_metadata_df['date_str'].isna()]
valid_rts_df = valid_rts_df[['alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID', 'OID', 'exact_date', 'broadcast_date', 
                             'broadcast_episode_title', 'broadcast_program_name',
                             'recording_place', 'series_title', 'content_summary', 'live', 
                             'work_duration', 'participants', 'radio_channels', 
                             'recording_dates', 'first_broadcast_dates']]
print(len(valid_rts_df))
valid_rts_df.head()

26225


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,OID,exact_date,broadcast_date,broadcast_episode_title,broadcast_program_name,recording_place,series_title,content_summary,live,work_duration,participants,radio_channels,recording_dates,first_broadcast_dates
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},True,True,"""X-Files"" ou ""Aux frontières du réel""",Analyse des médias,NaN,NaN,"Présentation de la série-culte. Le succès, com...",Non live,00:00:00.000,"[{'name': 'Frias, Roxanne', 'function': 'Inter...",['Espace 2'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996']
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},True,True,Internet à l'Ecole primaire d'Avully (GE),Analyse des médias,Avully,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Dubois, Laurent', 'function': 'Inte...",['Espace 2'],['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997']
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,{2AF13E50-C6A2-49E5-9851-06C441833B50},True,True,Le carré blanc,Analyse des médias,Genève;Fribourg,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Duparc, Nicole', 'function': 'Inter...",['Espace 2'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997']
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,{C940BD19-5503-4918-852B-D40C1C18B359},True,True,"Le feuilleton télévisé ""Urgences""",Analyse des médias,Genève,NaN,Analyse de la série et des raisons de son succès.,Non live,00:00:00.000,"[{'name': 'Kiefer, B.', 'function': 'Interview...",['Espace 2'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996']
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},True,True,Le journal à l'école,Analyse des médias,Genève;Fribourg,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Prophan, Geneviève', 'function': 'I...",['Espace 2'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997']


Finally we removed 36 broadcasts which were missing their date, ending up with 26225 radio issues

In [7]:
26261-26225

36

In [ ]:
"Valid�"

## index_file building (not final nor correct)

In [70]:
# Function to build issue index structure similar to issue_index.sub.json
def build_issue_index(documents, program_alias, program_dir):
    """
    Build an issue index structure from parsed documents.
    
    Structure:
    {
        "program_alias": {
            "year": {
                "month": [
                    {
                        "day": int,
                        "edition": str (a, b, c, etc.),
                        "local_path": str,
                        "audio_file": str or list,
                        ... other document metadata
                    },
                    ...
                ],
                ...
            },
            ...
        }
    }
    
    Args:
        documents: list of parsed document dictionaries
        program_alias: the program name/alias (e.g., "causerie_uni")
        program_dir: the path to the program directory
    
    Returns:
        tuple: (issue_index_dict, missing_audio_files_dict)
    """
    from datetime import datetime
    from collections import defaultdict
    
    issue_index = {program_alias: {}}
    missing_audio_files = {program_alias: []}
    
    # Group documents by recording date
    date_groups = defaultdict(list)
    
    for doc in documents:
        # Parse recording date (format: "DD/MM/YYYY - DD/MM/YYYY")
        recording_dates = doc.get('recordingdates', [])
        if not recording_dates:
            missing_audio_files[program_alias].append({
                'title': doc.get('TITLE'),
                'reason': 'No recording date'
            })
            continue
        
        date_str = recording_dates[0].split(' - ')[0]  # Get first date
        try:
            date_obj = datetime.strptime(date_str, "%d/%m/%Y")
            year = str(date_obj.year)
            month = f"{date_obj.month:02d}"
            day = date_obj.day
            date_key = (year, month, day)
            date_groups[date_key].append(doc)
        except:
            missing_audio_files[program_alias].append({
                'title': doc.get('TITLE'),
                'reason': f'Could not parse date: {date_str}'
            })
            continue
    
    # Build the nested structure
    for (year, month, day), docs_for_day in sorted(date_groups.items()):
        # Initialize year and month if not exist
        if year not in issue_index[program_alias]:
            issue_index[program_alias][year] = {}
        if month not in issue_index[program_alias][year]:
            issue_index[program_alias][year][month] = []
        
        # Assign editions (a, b, c, etc.) to documents on the same day
        for edition_idx, doc in enumerate(docs_for_day):
            edition = chr(ord('a') + edition_idx)  # 'a', 'b', 'c', ...
            
            # Extract audio files
            audio_files = []
            if doc.get('supports'):
                for support in doc['supports']:
                    filename = support.get('filename')
                    if filename:
                        audio_files.append(filename)
            
            # Create issue entry
            issue_entry = {
                'day': day,
                'edition': edition,
                'local_path': program_dir,
                'title': doc.get('TITLE'),
                'broadcast': doc.get('BROADCAST'),
                'summary': doc.get('SUMMARY'),
                'participants': doc.get('participants', []),
                'geographic_descriptors': doc.get('geographicaldescriptors', []),
                'thematic_descriptors': doc.get('thematicaldescriptors', []),
                'duration': doc.get('WORKDURATION'),
                'recording_dates': doc.get('recordingdates', [])
            }
            
            # Handle audio_file field (single value if 1 file, list if multiple)
            if not audio_files:
                # Missing audio files - add to separate tracking dict
                missing_audio_files[program_alias].append({
                    'title': doc.get('TITLE'),
                    'date': f"{day}/{month}/{year}",
                    'edition': edition,
                    'reason': 'No audio files in metadata'
                })
            elif len(audio_files) == 1:
                issue_entry['audio_file'] = audio_files[0]
            else:
                issue_entry['audio_file'] = audio_files
            
            issue_index[program_alias][year][month].append(issue_entry)
    
    return issue_index, missing_audio_files


# Test with the example documents
issue_index, missing_files = build_issue_index(all_documents, 'causerie_uni', '/mnt/project_impresso/original/RTS/causerie_uni')

print(f"Issue Index built successfully!")
print(f"Programs: {list(issue_index.keys())}")

for program, years_data in issue_index.items():
    print(f"\n{program}:")
    print(f"  Years: {sorted(years_data.keys())}")
    
    for year, months_data in sorted(years_data.items()):
        print(f"    {year}:")
        for month, issues in sorted(months_data.items()):
            print(f"      {month}: {len(issues)} issue(s)")
            for issue in issues[:2]:  # Show first 2
                audio_info = f"audio: {issue.get('audio_file', 'N/A')}"
                if isinstance(issue.get('audio_file'), list):
                    audio_info = f"audio: {len(issue['audio_file'])} files"
                print(f"        Day {issue['day']}-{issue['edition']}: {audio_info}")

print(f"\n\nMissing audio files:")
for program, missing in missing_files.items():
    print(f"{program}: {len(missing)} document(s) with missing audio")
    for m in missing[:3]:  # Show first 3
        print(f"  - {m.get('title', 'N/A')[:60]}... ({m.get('reason', 'N/A')})")


Issue Index built successfully!
Programs: ['causerie_uni']

causerie_uni:
  Years: ['1939', '1940', '1942']
    1939:
      11: 6 issue(s)
        Day 4-a: audio: 1204286700-1542X_p_complet_wav958-SIROM{A3020B9E-9A94-4DFD-B1E0-CC34FE2F13AC}.wav
        Day 20-a: audio: 1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.wav
    1940:
      11: 1 issue(s)
        Day 26-a: audio: 1209129877-3151_p_complet_wav_958-SIROM{8357777C-7412-4674-98A8-B3AC6AB572DB}.wav
    1942:
      10: 1 issue(s)
        Day 3-a: audio: N/A


Missing audio files:
causerie_uni: 2 document(s) with missing audio
  - Le Tessin, facteur de coh?sion nationale : Causerie de Giova... (No audio files in metadata)
  - Epicure ou la religion du plaisir : Causerie de Ren? Schaere... (No audio files in metadata)


In [71]:
issue_index

{'causerie_uni': {'1939': {'11': [{'day': 4,
     'edition': 'a',
     'local_path': '/mnt/project_impresso/original/RTS/causerie_uni',
     'title': "L'enseignement de l'histoire. Causerie de Gonzague de Reynold",
     'broadcast': 'Causerie universitaire',
     'summary': "Cr?ation ? l'Universit? de Fribourg de la Chaire d'histoire de la civilisation moderne. L'enseignement de l'histoire a pour but d'initier les auditeurs ? la vie, de leur donner les connaissances et la compr?hension de l'Europe actuelle et tragique. L'?ducation de la pens?e doit d?velopper le sens critique en multipliant les termes de comparaison. La m?connaissance de l'histoire comme sympt?me de l'inculture d'une grande partie de la jeunesse. Devenir contemporain du pass? pour mieux le comprendre et appr?hender le pr?sent ? la lumi?re des origines.",
     'participants': [{'name': 'Reynold, Gonzague de',
       'function': 'Conf?rencier/e',
       'role': '?crivain'}],
     'geographic_descriptors': ['Fribourg (can